# Power System Load Type Classification

**Goal:** predict `Load_Type` (Light / Medium / Maximum load) of an industrial power system from historical readings.

**Pipeline:** data loading → EDA → feature engineering → time-based validation → model comparison → final evaluation on the **last month** (held-out test set).

**Key design decisions**
- Data is a time series, so the split is **chronological** (no random shuffling). The last calendar month is the test set.
- Model selection uses **TimeSeriesSplit** on the training portion only, so the test set is never touched until the end.
- Evaluation: accuracy, precision, recall, F1 (macro and per class), confusion matrix.

In [ ]:
import warnings, glob, os
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Load data
Place `load_data.csv` next to this notebook (or change `DATA_PATH`).

In [ ]:
DATA_PATH = "load_data.csv"

if DATA_PATH is None:
    found = sorted(glob.glob("*.csv"))
    assert found, "Put the dataset CSV next to this notebook or set DATA_PATH."
    DATA_PATH = found[0]
print("Using:", DATA_PATH)

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

### Standardise column names
Column names differ slightly between versions of the dataset, so they are mapped to a fixed set by keyword.

In [ ]:
def rename_cols(cols):
    out = {}
    for c in cols:
        k = c.lower().replace(" ", "_")
        if k in ("date", "date_time", "datetime"):  out[c] = "Date"
        elif k.startswith("usage"):             out[c] = "Usage_kWh"
        elif "lagging" in k and "factor" not in k: out[c] = "Lagging_kVarh"
        elif "leading" in k and "factor" not in k: out[c] = "Leading_kVarh"
        elif "lagging" in k and "factor" in k:  out[c] = "Lagging_PF"
        elif "leading" in k and "factor" in k:  out[c] = "Leading_PF"
        elif k.startswith("co2"):               out[c] = "CO2"
        elif k == "nsm":                        out[c] = "NSM"
        elif k.replace("_", "") == "loadtype":  out[c] = "Load_Type"
    return out

df = df.rename(columns=rename_cols(df.columns))
df["Load_Type"] = df["Load_Type"].astype(str).str.strip().str.replace(" ", "_")

# Dates in this dataset are day-first (dd/mm/yyyy hh:mm)
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)

print(df.dtypes, "\n")
print("Date range:", df["Date"].min(), "->", df["Date"].max())

## 2. Data quality check and cleaning
The raw file is not clean. Profiling shows:
- **Missing values** in almost every numeric column.
- **Corrupted readings:** genuine meter values have at most 2 decimals, but about 10% of values in each numeric column carry long random decimals (e.g. `8.7536924...`) and are often wildly off (usage up to 435 kWh versus a real maximum near 153). These are treated as bad readings.
- **Impossible values:** power factors above 100%.
- **`NSM` does not match the timestamp** for the same corrupted rows. It is fully determined by `Date`, so it is recomputed.
- **One duplicate timestamp** (31-12-2018 20:00).

Cleaning is row-wise (no information flows between rows), so it cannot leak test data into training. Bad readings become `NaN` and are imputed later *inside the model pipelines*, fitted on training data only.

In [ ]:
raw_n = len(df)
df = df.drop_duplicates(subset="Date", keep="first").reset_index(drop=True)
print("Dropped duplicate timestamps:", raw_n - len(df))

num_raw = [c for c in ["Usage_kWh", "Lagging_kVarh", "Leading_kVarh", "CO2", "Lagging_PF", "Leading_PF"] if c in df.columns]

report = {}
for c in num_raw:
    before_missing = df[c].isna().sum()
    # genuine readings have <= 2 decimals; longer decimals mark a corrupted value
    corrupted = df[c].notna() & ((df[c] - df[c].round(2)).abs() > 1e-6)
    df.loc[corrupted, c] = np.nan
    n_corrupt = int(corrupted.sum())
    n_range = 0
    if c.endswith("_PF"):                        # power factor must be within 0-100 %
        out_of_range = df[c] > 100
        n_range = int(out_of_range.sum())
        df.loc[out_of_range, c] = np.nan
    report[c] = {"originally_missing": int(before_missing), "corrupted_to_nan": n_corrupt,
                 "out_of_range_to_nan": n_range, "missing_after_cleaning": int(df[c].isna().sum())}

# NSM is fully determined by the timestamp: rebuild it rather than trust the raw column
nsm_wrong = (df["NSM"] != (df["Date"].dt.hour * 3600 + df["Date"].dt.minute * 60 + df["Date"].dt.second)).sum()
df["NSM"] = df["Date"].dt.hour * 3600 + df["Date"].dt.minute * 60 + df["Date"].dt.second
print("NSM values wrong or missing in raw data, rebuilt from Date:", nsm_wrong)
pd.DataFrame(report).T

## 3. Exploratory data analysis

In [ ]:
print("Missing values:\n", df.isna().sum(), "\n")
print("Duplicate rows:", df.duplicated().sum(), "| duplicate timestamps:", df["Date"].duplicated().sum())
df.describe().T

In [ ]:
order = ["Light_Load", "Medium_Load", "Maximum_Load"]
counts = df["Load_Type"].value_counts().reindex(order)
print(counts, "\n", (counts / counts.sum()).round(3))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
counts.plot.bar(ax=ax[0], color=["#8fb8de", "#f2c14e", "#d1495b"], title="Class distribution")
df.groupby([df["Date"].dt.to_period("M"), "Load_Type"]).size().unstack().reindex(columns=order).plot.bar(
    stacked=True, ax=ax[1], color=["#8fb8de", "#f2c14e", "#d1495b"], title="Classes per month")
plt.tight_layout(); plt.show()

In [ ]:
num_cols = [c for c in ["Usage_kWh", "Lagging_kVarh", "Leading_kVarh", "CO2", "NSM"] if c in df.columns]
fig, axes = plt.subplots(1, len(num_cols), figsize=(4 * len(num_cols), 3.5))
for a, c in zip(np.atleast_1d(axes), num_cols):
    data = [df.loc[df.Load_Type == t, c] for t in order]
    a.boxplot(data, labels=["Light", "Medium", "Max"], showfliers=False)
    a.set_title(c)
plt.tight_layout(); plt.show()

print(df[num_cols].corr().round(2))

In [ ]:
# Typical daily profile: how load type depends on time of day
tmp = df.assign(hour=df["NSM"] // 3600)
prof = pd.crosstab(tmp["hour"], tmp["Load_Type"], normalize="index").reindex(columns=order)
prof.plot.area(figsize=(10, 3.5), color=["#8fb8de", "#f2c14e", "#d1495b"], title="Share of each load type by hour of day")
plt.show()

In [ ]:
# Dominant load type for every (day of week, time slot), split into winter (Nov-Feb) and the rest of the year
code_map = {"Light_Load": 0, "Medium_Load": 1, "Maximum_Load": 2}
t = df.assign(slot=(df["NSM"] // 900).astype(int), dow=df["Date"].dt.dayofweek,
              winter=df["Date"].dt.month.isin([11, 12, 1, 2]), y=df["Load_Type"].map(code_map))
from matplotlib.colors import ListedColormap
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), sharey=True)
for a, (flag, title) in zip(axes, [(True, "Winter (Nov-Feb)"), (False, "Mar-Oct")]):
    grid = t[t.winter == flag].groupby(["dow", "slot"])["y"].agg(lambda s: s.mode()[0]).unstack()
    a.imshow(grid.values, aspect="auto", cmap=ListedColormap(["#8fb8de", "#f2c14e", "#d1495b"]), vmin=0, vmax=2,
             extent=[0, 24, 6.5, -0.5])
    a.set_title(f"Most common load type - {title}"); a.set_xlabel("hour of day"); a.set_xticks(range(0, 25, 4))
axes[0].set_yticks(range(7)); axes[0].set_yticklabels(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
plt.tight_layout(); plt.show()

# Within a fixed daytime slot, how well does usage separate Medium from Maximum?
slot = t[(t.dow < 5) & (~t.winter) & (t.slot.between(40, 63)) & (t.y > 0)]
print("Usage (kWh) on Mar-Oct weekdays, 10:00-16:00, Medium vs Maximum:")
print(slot.groupby("Load_Type")["Usage_kWh"].describe().round(1))

# Days that break the schedule (weekday that is Light all day = public holiday)
daily = df.assign(day=df["Date"].dt.date, wd=df["Date"].dt.dayofweek < 5)
share = daily[daily.wd].groupby("day")["Load_Type"].apply(lambda s: (s != "Light_Load").mean())
print("\nWeekdays with (almost) no Medium/Max load - likely holidays:", int((share < 0.1).sum()))

**EDA takeaways**
- The classes are imbalanced (Light is the largest), so accuracy alone is misleading and macro-F1 is the main selection metric.
- **`Load_Type` follows a time-of-use schedule**: Light overnight and all day Sunday, Medium and Maximum in fixed windows on weekdays and Saturday. The Maximum window is different in winter (Nov-Feb) than in the rest of the year, and public holidays behave like Sundays.
- `Usage_kWh`, `Lagging_kVarh` and `CO2` are strongly correlated with each other. Usage helps to separate Light from the rest, but Medium and Maximum overlap heavily in usage within the same time slot, so the **clock, weekday, season and holiday** are the most informative signals.

## 4. Feature engineering
- **Time features** from `Date` / `NSM`: hour, cyclical sin/cos of time of day (so 23:45 is close to 00:00), day of week, weekend flag, month.
- **Calendar features from the EDA:** season flag (winter vs rest of year, the tariff window changes with it) and the 15-minute slot of the day.
- **Holiday signal:** `day_usage_mean`, the average usage of the same calendar day. It uses only readings from the same day (not other days), and is checked in an ablation below.
- **Ratio features**: `Lagging_kVarh / Usage_kWh`, `Leading_kVarh / Usage_kWh`, and net reactive power.
- Lag/rolling features use only **past** values (last good reading, then `shift(1)`), so no future information leaks into a row.

In [ ]:
def make_features(d):
    d = d.copy()
    d["hour"] = d["NSM"] // 3600
    d["tod_sin"] = np.sin(2 * np.pi * d["NSM"] / 86400)
    d["tod_cos"] = np.cos(2 * np.pi * d["NSM"] / 86400)
    d["dow"] = d["Date"].dt.dayofweek
    d["is_weekend"] = (d["dow"] >= 5).astype(int)
    d["month"] = d["Date"].dt.month
    d["is_winter"] = d["month"].isin([11, 12, 1, 2]).astype(int)   # season the tariff window changes with
    d["slot"] = d["NSM"] // 900                                      # 15-minute slot of the day (0-95)

    eps = 1e-6
    d["lag_ratio"] = d["Lagging_kVarh"] / (d["Usage_kWh"] + eps)
    d["lead_ratio"] = d["Leading_kVarh"] / (d["Usage_kWh"] + eps)
    d["net_reactive"] = d["Lagging_kVarh"] - d["Leading_kVarh"]

    # past-only context (shift(1) excludes the current row)
    u = d["Usage_kWh"].ffill()                    # last known good reading (past only)
    d["usage_prev"] = u.shift(1)
    d["usage_roll4_mean"] = u.shift(1).rolling(4, min_periods=1).mean()
    d["usage_diff"] = d["Usage_kWh"] - d["usage_prev"]

    # same-day context: average usage of that calendar day (helps to spot holidays, which run like Sundays)
    d["day_usage_mean"] = d.groupby(d["Date"].dt.date)["Usage_kWh"].transform("mean")
    return d

feat_df = make_features(df)

FEATURES = [c for c in ["Usage_kWh", "Lagging_kVarh", "Leading_kVarh", "CO2", "Lagging_PF", "Leading_PF", "NSM",
                        "tod_sin", "tod_cos", "slot", "dow", "is_weekend", "is_winter", "day_usage_mean",
                        "lag_ratio", "lead_ratio", "net_reactive",
                        "usage_prev", "usage_roll4_mean", "usage_diff"] if c in feat_df.columns]
print(len(FEATURES), "features:", FEATURES)

## 5. Train / test split (time-based)
The **last calendar month** in the data is the test set; everything before it is training data.

In [ ]:
last_period = feat_df["Date"].dt.to_period("M").max()
test_mask = feat_df["Date"].dt.to_period("M") == last_period

train, test = feat_df[~test_mask], feat_df[test_mask]
X_train, y_train = train[FEATURES], train["Load_Type"]
X_test,  y_test  = test[FEATURES],  test["Load_Type"]

print("Test month:", last_period)
print("Train:", X_train.shape, train["Date"].min(), "->", train["Date"].max())
print("Test :", X_test.shape,  test["Date"].min(),  "->", test["Date"].max())
print("\nTrain class share:\n", y_train.value_counts(normalize=True).round(3))
print("\nTest class share:\n", y_test.value_counts(normalize=True).round(3))

## 6. Model selection with time-series cross-validation
Candidates: a scaled logistic regression (baseline), a random forest and a histogram gradient boosting model. Remaining `NaN`s are median-imputed inside the pipeline (fitted on training folds only); gradient boosting handles `NaN` natively. Cross-validation uses expanding-window `TimeSeriesSplit` **inside the training set only**. Class weights handle the imbalance.

In [ ]:
models = {
    "Logistic Regression": Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()),
                                     ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))]),
    "Random Forest": Pipeline([("imp", SimpleImputer(strategy="median")),
                               ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight="balanced",
                                                              n_jobs=-1, random_state=RANDOM_STATE))]),
    "Hist Gradient Boosting": HistGradientBoostingClassifier(learning_rate=0.08, max_depth=6,
                                                             class_weight="balanced", random_state=RANDOM_STATE),
}

tscv = TimeSeriesSplit(n_splits=5)
cv_rows = []
for name, m in models.items():
    f1 = cross_val_score(m, X_train, y_train, cv=tscv, scoring="f1_macro", n_jobs=-1)
    acc = cross_val_score(m, X_train, y_train, cv=tscv, scoring="accuracy", n_jobs=-1)
    cv_rows.append({"model": name, "cv_macro_f1": f1.mean(), "cv_f1_std": f1.std(), "cv_accuracy": acc.mean()})
cv_results = pd.DataFrame(cv_rows).sort_values("cv_macro_f1", ascending=False).round(4)
cv_results

## 7. Final evaluation on the held-out last month

In [ ]:
rows, fitted = [], {}
for name, m in models.items():
    m.fit(X_train, y_train)
    fitted[name] = m
    pred = m.predict(X_test)
    p, r, f, _ = precision_recall_fscore_support(y_test, pred, average="macro", zero_division=0)
    rows.append({"model": name, "accuracy": accuracy_score(y_test, pred),
                 "precision_macro": p, "recall_macro": r, "f1_macro": f})
test_results = pd.DataFrame(rows).sort_values("f1_macro", ascending=False).round(4)
test_results

In [ ]:
# Best model = highest cross-validated macro-F1 (chosen on training data only, not on the test set)
best_name = cv_results.iloc[0]["model"]
best = fitted[best_name]
pred = best.predict(X_test)
print("Selected model:", best_name, "\n")
print(classification_report(y_test, pred, labels=order, digits=3, zero_division=0))

cm = confusion_matrix(y_test, pred, labels=order)
ConfusionMatrixDisplay(cm, display_labels=["Light", "Medium", "Max"]).plot(cmap="Blues", values_format="d")
plt.title(f"{best_name} - confusion matrix (test month {last_period})"); plt.show()

### Ablation and error analysis
Two checks: (1) does the same-day `day_usage_mean` feature matter, and (2) on which dates does the model go wrong?

In [ ]:
from sklearn.base import clone
def test_scores(feats):
    m = clone(models[best_name]).fit(X_train[feats], y_train)
    p = m.predict(X_test[feats])
    return accuracy_score(y_test, p), precision_recall_fscore_support(y_test, p, average="macro", zero_division=0)[2]

no_day = [f for f in FEATURES if f != "day_usage_mean"]
no_usage = [f for f in FEATURES if f not in ("Usage_kWh", "usage_prev", "usage_roll4_mean", "usage_diff", "day_usage_mean", "CO2", "net_reactive", "lag_ratio", "lead_ratio")]
time_only = ["NSM", "tod_sin", "tod_cos", "slot", "dow", "is_weekend", "is_winter"]
ab = pd.DataFrame({"all features": test_scores(FEATURES), "without day_usage_mean": test_scores(no_day),
                   "calendar/time features only": test_scores(time_only)}, index=["accuracy", "macro_F1"]).T.round(4)
ab

In [ ]:
err = test.assign(pred=pred)
err = err[err["pred"] != err["Load_Type"]]
print("Misclassified rows:", len(err), "of", len(test))
print(err.groupby(err["Date"].dt.date).size().sort_values(ascending=False).head(8).rename("errors per day"))

## 8. Interpretation

In [ ]:
from sklearn.inspection import permutation_importance
if True:
    r = permutation_importance(best, X_test, y_test, scoring="f1_macro", n_repeats=3,
                               random_state=RANDOM_STATE, n_jobs=-1)
    imp = pd.Series(r.importances_mean, index=FEATURES)
imp.sort_values().tail(12).plot.barh(figsize=(7, 4), title=f"Permutation importance (macro-F1) - {best_name}")
plt.tight_layout(); plt.show()

## 9. Conclusions
- A chronological split with the last month held out gives an honest estimate of performance on recent, unseen data.
- Model choice was made on time-series CV over training data only; the test month was used once for the final report.
- Energy usage and reactive power dominate predictions, with time of day adding context for borderline cases.
- Most remaining errors sit between neighbouring classes (Light↔Medium, Medium↔Maximum), as the confusion matrix shows.

**Limitations / next steps:** a single test month can be affected by seasonality, so results should be confirmed with rolling-origin evaluation over several months; hyperparameter tuning (e.g. Optuna) and probability calibration are natural extensions.